# Match the chl dataset to global OWT data

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import math
from math import radians, sin, cos, sqrt, atan2
import xarray as xr
from datetime import datetime,date, timedelta
import datetime as dt 
import time
from scipy import stats
import os

In [ ]:
def haversine_distance_vectorized(lat1, lon1, lat2, lon2):
    """
    Calculate the haversine distance between two sets of points in a vectorized way.
    Parameters:
        lat1, lon1: Latitude and longitude of the first set of points (arrays).
        lat2, lon2: Latitude and longitude of the second set of points (arrays).
    Returns:
        Array of distances in kilometers.
    """
    R = 6371  #radius of earth
    #convert latitude and longitude from degrees to radians
    lat1_rad, lon1_rad = np.radians(lat1), np.radians(lon1)
    lat2_rad, lon2_rad = np.radians(lat2), np.radians(lon2)

    #calculate differences in coordinates
    dlon = lon2_rad - lon1_rad
    dlat = lat2_rad - lat1_rad

    #haversine formula
    a = np.sin(dlat / 2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    distance = R * c
    return distance

In [ ]:
def insitu_year(df,year):
    ''' 
    Takes a dataframe and given year, returns dataframe with only data from that year and list of YYYmmdd times to pull xarray file names 
    df = in situ dataframe, with datetime column
    year = desired year
    returns:
    filtered_df= in situ dataframe subset to the given year
    dates = list of unique dates in format YYYYmmdd
    '''
    year = str(year) #ensure year is string
    start_date = pd.Timestamp(year+'-01-01 00:00:00') #jan 1st of year
    end_date = pd.Timestamp(year+'-12-31 23:59:59') #dec 31st of year
    filtered_df = df[df['datetime'].between(start_date,end_date)].reset_index(drop=True) #only pull between these dates from df
    unique_dates=filtered_df.copy()
    unique_dates['date'] = unique_dates.datetime.dt.date #create column of just dates without times
    unique_dates = unique_dates.drop_duplicates(subset=['date'], keep='first').astype(str) #find all unique dates and turn to str
    unique_dates['date'] = unique_dates['date'].str.replace('-', '') #take out all - 
    dates = unique_dates.date #return just the dates as an array 
    return filtered_df, dates

In [ ]:
def sat_insit_match3(sat_xr, insit_df, spatial_diff_km,variable):
    ''' takes a satellite xarray and a in situ dataframe and finds datapoints with same lat/lon/datetime withing bounds
        sat_xr: satellite satellite xarray subset to only unique dates from in situ dataframe
        insit_df: in situ dataframe subset to specific year 
        spatial_diff_km: spatial boundary for matching in km
        var: satellite variable name to match
        return: insitu dataframe with additional colum for satellite matches 
    '''
    #First make sure the in situ dataset has the right datetime format 
    insit_df['datetime'] = pd.to_datetime(insit_df['datetime'], errors='coerce')
    insit_df.dropna(subset=['datetime'], inplace=True) #remove any empty datetime rows
    insit_df.dropna(subset=['lat'], inplace=True) #remove rows without complete coordinates
    insit_df.reset_index(inplace=True, drop=True)
    
    time_indexer = xr.DataArray(insit_df['datetime'], dims='point') #pull all the dates and turn into array
    lat_indexer = xr.DataArray(insit_df['lat'], dims='point') #pull all lats and turn into an array
    lon_indexer = xr.DataArray(insit_df['lon'], dims='point') #pull all lons and turn into an array 

    print("Finding nearest satellite points for all in-situ data at once...")
    #use xarray.sel to gather the same lat, lon, and time points as the in situ dataset from the satellite data
    matched_sat_points = sat_xr.sel(time=time_indexer, latitude=lat_indexer,longitude=lon_indexer, method='nearest')

    print("Filtering matches based on spatial and temporal criteria...")
    #calculate the distance in the matched sat point is from the in situ point.
    distances_km = haversine_distance_vectorized(insit_df['lat'].values, insit_df['lon'].values, matched_sat_points['latitude'].values,
                                                matched_sat_points['longitude'].values)

    #check if the match is on the exact same day.
    time_match_mask = insit_df['datetime'].dt.date.values == pd.to_datetime(matched_sat_points['time'].values).date
    
    #check if the match is within 6km of the in situ coordinates 
    spatial_match_mask = distances_km <= spatial_diff_km

    #combine the two checks to create a mask where the dates match and within 6kms
    valid_match_mask = time_match_mask & spatial_match_mask

    #initiate columns to put satellite information 
    insit_df['owt'] = np.nan
    insit_df['all_classes'] = np.nan
    insit_df['all_classes'] = insit_df['all_classes'].astype(object)

    #create mask that only retains data where the dates match. 
    valid_indices = np.where(valid_match_mask)[0]
    print(f"Found {len(valid_indices)} valid matches out of {len(insit_df)} points. Now extracting 3x3 grids...")
    
    #Gather surrounding 3x3 pixels and find the mode 
    if len(valid_indices) > 0: #as long as there are points that pass the date mask
        sat_lat_coords = sat_xr['latitude'].values #grab the lat and lon points from WHOLE xarray
        sat_lon_coords = sat_xr['longitude'].values

        #loop ONLY over the valid matches to extract the 3x3 grid.
        for idx in valid_indices:
            #get the coordinates of the valid matched satellite point.
            matched_lat = matched_sat_points['latitude'].values[idx]
            matched_lon = matched_sat_points['longitude'].values[idx]
            matched_time = matched_sat_points['time'].values[idx]

            #using the coordinates from the valid matches, gather the index value that matches to the whole xarray
            lat_idx = sat_xr.indexes['latitude'].get_loc(matched_lat)
            lon_idx = sat_xr.indexes['longitude'].get_loc(matched_lon)
            time_idx = sat_xr.indexes['time'].get_loc(matched_time)

            #gather slice information the surrounding lat and lon pixels 
            lat_slice = slice(max(0, lat_idx - 1), min(len(sat_lat_coords), lat_idx + 2))
            lon_slice = slice(max(0, lon_idx - 1), min(len(sat_lon_coords), lon_idx + 2))
            
            #extract the 3x3 grid using isel.
            grid_data = sat_xr[variable].isel(time=time_idx, latitude=lat_slice, longitude=lon_slice) #gather the OWT data from surrounding pixels
            grid_values = grid_data.values.flatten() #flatten array
            
            #calculate the mode, ignoring NaN values.
            valid_grid_values = grid_values[~np.isnan(grid_values)] #remove any nans from the array
            grid_mode = stats.mode(valid_grid_values).mode if valid_grid_values.size > 0 else np.nan #as long as the array has a value, find the mode, else set as nan

            #store the results in the DataFrame.
            insit_df.loc[idx, 'owt'] = grid_mode
            insit_df.at[idx, 'all_classes'] = grid_values.tolist()

    print('Finished matching process!')
    return insit_df

In [ ]:
def slice_before_concat(ds):
        """
        Slices the xarray dataset to keep only a specific lat/lon range instead of loading in global data
        NOTE: 'lat' and 'lon' here need to have same naming convention as the satellite nc file
        """
        ds = ds.sel(latitude=slice(80,10),longitude=slice(-179,-40)) #slice xarray within these lat and lon bounds 
        return ds

In [ ]:
#def loop for subsetting in situ and satellite xarray into correct years and running matching code. 
def owt_insit_match(insitu_df, sat_xr, spatial_res, variable, year):
    '''
    code that produces dataframe of matching satellite points and insitu points.
    input: insitu_df: dataframe with datetime, lat, lon, and variables
        sat_xr: folder string to satellite data files
        spatial_res: resolution boundry for matching (km)
        variable: the variable being matched (OWT)
        year: the year in situ and sat data are being compared (int)
    returns:
        in situ dataframe with matched datasets subset to the given year 
    '''
    #first, reduce insitu_df to only stated year 
    insit_year, dates_year= insitu_year(insitu_df,year)

    #next, pull xarray files that match to the dates of the insit_year
    existing_files= []
    for date in sorted(dates_year):
        file_date = str(sat_xr)+str(date)+'-OCCCI-V6.0-GLOBAL_MAPPED-OWC_WEI.nc'
        if os.path.exists(file_date):
            existing_files.append(file_date)
        else:
            print(f"Warning: File not found for date {date}: {file_date}")
            
    ## NOTE: uncomment below if you're getting errors about corrupted files. THis code will open each file to ensure there's actual data inside
    # existing_files= []
    # var = 'water_class'
    # for date in sorted(dates_year):
    #    file_date = str(sat_xr)+str(date)+'-OCCCI-V6.0-GLOBAL_MAPPED-OWC_WEI.nc'
    #    if os.path.exists(file_date): #if path exists, try to open it
    #        try:
    #            temp_ds = xr.open_dataset(file_date, chunks={}) # Use chunks={} for Dask if needed
    #            _ = temp_ds[var].isel(time=0, latitude=0, longitude=0).compute()
    #            temp_ds.close()
    #           existing_files.append(file_date)
    #        except Exception as e:
    #            print(f"Skipping corrupted or unreadable file: {file_date}") #. Error: {e}
                #temp_ds.close()
    #    else:
    #         print(f"Warning: File not found for date {date}: {file_date}")

    
    #concat all satellite .nc files into 1 single xarray file       
    coastal_us_combo= xr.open_mfdataset(existing_files, preprocess=slice_before_concat,combine='by_coords') #open all files, concat into one subset to north america
    print('satellite refiguring complete')
    test_insitu = insit_year.copy()
    matched_chl = sat_insit_match3(coastal_us_combo, test_insitu, spatial_res,variable) #run the satellite matching function
    return matched_chl  #matched_chl = dataframe of in situ chlorophyll to OC-CCI OWT 


In [ ]:
#load in all the data with the sattellite matchups 
chl_all = pd.read_excel(r"Coastal_chl_final\sat_matchupCHL2.xlsx")
chl_all['datetime'] = pd.to_datetime(chl_all['datetime'],format='mixed') 

In [ ]:
#all reducing already done in first round of matchups 
insit_chl_coast2 = chl_all.copy()
test= insit_chl_coast2['datetime'].dt.year
test.value_counts()

In [ ]:
#to match, run matching code for each year

In [ ]:
df_matched2000 = owt_insit_match(insit_chl_coast2, r'OWC_WEI\D_',6, 'water_class', 2000)

In [ ]:
df_matched2001 = owt_insit_match(insit_chl_coast2, r'OWC_WEI\D_',6, 'water_class', 2001)

In [ ]:
df_matched2002 = owt_insit_match(insit_chl_coast2, r'OWC_WEI\D_',6, 'water_class', 2002)

In [ ]:
df_matched2003 = owt_insit_match(insit_chl_coast2, r'OWC_WEI\D_',6, 'water_class', 2003)

In [ ]:
df_matched2004 = owt_insit_match(insit_chl_coast2, r'OWC_WEI\D_',6, 'water_class', 2004)

In [ ]:
df_matched2005 = owt_insit_match(insit_chl_coast2, r'OWC_WEI\D_',6, 'water_class', 2005)

In [ ]:
df_matched2006 = owt_insit_match(insit_chl_coast2, r'OWC_WEI\D_',6, 'water_class', 2006)

In [ ]:
df_matched2007 = owt_insit_match(insit_chl_coast2, r'OWC_WEI\D_',6, 'water_class', 2007)

In [ ]:
df_matched2008 = owt_insit_match(insit_chl_coast2, r'OWC_WEI\D_',6, 'water_class', 2008)

In [ ]:
df_matched2009 = owt_insit_match(insit_chl_coast2, r'OWC_WEI\D_',6, 'water_class', 2009)

In [ ]:
df_matched2010 = owt_insit_match(insit_chl_coast2, r'OWC_WEI\D_',6, 'water_class', 2010)

In [ ]:
df_matched2011 = owt_insit_match(insit_chl_coast2, r'OWC_WEI\D_',6, 'water_class', 2011)

In [ ]:
df_matched2012 = owt_insit_match(insit_chl_coast2, r'OWC_WEI\D_',6, 'water_class', 2012)

In [ ]:
df_matched2013 = owt_insit_match(insit_chl_coast2, r'OWC_WEI\D_',6, 'water_class', 2013)

In [ ]:
df_matched2014 = owt_insit_match(insit_chl_coast2, r'OWC_WEI\D_',6, 'water_class', 2014)

In [ ]:
df_matched2015 = owt_insit_match(insit_chl_coast2, r'OWC_WEI\D_',6, 'water_class', 2015)

In [ ]:
df_matched2016 = owt_insit_match(insit_chl_coast2, r'OWC_WEI\D_',6, 'water_class', 2016)

In [ ]:
df_matched2017 = owt_insit_match(insit_chl_coast2, r'OWC_WEI\D_',6, 'water_class', 2017)

In [ ]:
df_matched2018 = owt_insit_match(insit_chl_coast2, r'OWC_WEI\D_',6, 'water_class', 2018)

In [ ]:
df_matched2019 = owt_insit_match(insit_chl_coast2, r'OWC_WEI\D_',6, 'water_class', 2019)

In [ ]:
df_matched2020 = owt_insit_match(insit_chl_coast2, r'OWC_WEI\D_',6, 'water_class', 2020)

In [ ]:
df_matched2021 = owt_insit_match(insit_chl_coast2, r'OWC_WEI\D_',6, 'water_class', 2021)

In [ ]:
df_matched2022 = owt_insit_match(insit_chl_coast2, r'OWC_WEI\D_',6, 'water_class', 2022)

In [ ]:
df_matched2023 = owt_insit_match(insit_chl_coast2, r'OWC_WEI\D_',6, 'water_class', 2023)

In [ ]:
df_matched2024 = owt_insit_match(insit_chl_coast2, r'OWC_WEI\D_',6, 'water_class', 2024)

In [ ]:
df_matched2025 = owt_insit_match(insit_chl_coast2, r'OWC_WEI\D_',6, 'water_class', 2025)

In [ ]:
current_matchup = pd.read_excel(r'Coastal_chl_final\owt_allchlV3.xlsx')

In [ ]:
dfs=[current_matchup,df_matched2023] 
matched_all = pd.concat(dfs).reset_index(drop=True)

In [ ]:
matched_all.to_excel("owt_allchlV3.xlsx", index = False)